# Time series analysis
---

Running this notebook you will:

- Visualize the timeseries
- Extract the temporale and frequency features
- Plot the univarialte and bivariate relation between the features

In [72]:
from ipywidgets import widgets, Layout
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objs as go
from IPython.display import display, display_markdown, HTML


import glob
import numpy as np
import os
import pandas as pd
from pathlib import Path
import sys
import time
import threading

sys.path.append('../utils/')
from pre_processing import process_single_ts
from feature_extraction import create_feature_in_dataframe, extract_acf_values

## Step 1. Custom data path

In case you change the folder structure, you can insert here the paths to the data (extracted time series) and the figures. You can also select the working datasheet you want to use.

In [73]:
data_path = widgets.Text(
    value='extracted_signals',
    placeholder='Insert path here',
    description='data_path',
    disabled=True,
    style= {'description_width': 'initial'},
    layout=widgets.Layout(width='95%')
)
checkbox_data_path = widgets.Checkbox(
    value=False,
    description='edit path',
    disabled=False,
    indent=False
)
def data_path_change(c):
    if checkbox_data_path.value:
        data_path.disabled = False
    else:
        data_path.disabled = True

checkbox_data_path.observe(data_path_change, names="value")

figure_path = widgets.Text(
    value='figures',
    placeholder='Insert path here',
    description='figure_path',
    disabled=True,
    style= {'description_width': 'initial'},
    layout=widgets.Layout(width='95%')
)
checkbox_figure_path = widgets.Checkbox(
    value=False,
    description='edit path',
    disabled=False,
    indent=False
)
def figure_path_change(c):
    if checkbox_data_path.value:
        figure_path.disabled = False
    else:
        figure_path.disabled = True

checkbox_figure_path.observe(figure_path_change, names="value")

files = [file for file in os.listdir('.') if (file.endswith('.xlsx') or file.endswith('.csv') or file.endswith('.xls'))]
excel_file = widgets.Combobox(
    options=files,
    placeholder='Select or type a file',
    description='Working Datasheet:',
    style= {'description_width': 'initial'},
    layout=widgets.Layout(width='95%')
)

button_excel_file_status = widgets.Button(
    description='',
    disabled=True,
    button_style='',
    layout=widgets.Layout(width='5%')
)

def path_change(b=None):
    if excel_file.value == '':
        button_excel_file_status.button_style = ''
        button_excel_file_status.icon = ' '
        button_preprocess_data.disabled = True
    elif os.path.isfile(excel_file.value):
        button_excel_file_status.button_style = 'success'
        button_excel_file_status.icon = 'check'
        button_preprocess_data.disabled = False
    else:
        button_excel_file_status.button_style = 'danger'
        button_excel_file_status.icon = 'times-circle'
        button_preprocess_data.disabled = True

excel_file.observe(path_change, names="value")
#interactive_widgets_path = widgets.interactive(path_change)

In [74]:
display(widgets.HBox([data_path, checkbox_data_path]))
display(widgets.HBox([figure_path, checkbox_figure_path]))
display(widgets.HBox([excel_file, button_excel_file_status]))

## Step 2. Preprocessing

The pre-processing consists of the following steps:
- polynomial interpolation in case missing values are present
- de-trending
- smoothing
- scaling: pixel in micrometer and bin in seconds

Click on the ```Preprocess data``` button to start print the list of preprocessed files and perform the preprocessing step.

In [75]:
out_process_data = widgets.Output()
filename_list = []
data_processed = []
data_info = None

@out_process_data.capture(clear_output=True)
def preprocess_data(b=None):
    global filename_list
    global data_processed
    global data_info

    button_preprocess_data.disabled = True
    # Open data info file
    data_info = pd.read_excel(excel_file.value)
    # Rename columns and fill NA values
    data_info.rename(columns={'File_video name': 'Video name'}, inplace=True)
    data_info.fillna({'Treatment during contraction': 'Other'}, inplace=True)
    
    # Open time series files
    filename_list = []
    data = []
    files = []
    
    for filename in glob.glob(os.path.join(data_path.value, '*.npy')):
        filename_list.append(Path(filename).stem)
        with open(filename, 'r') as f:
            data.append([Path(filename).stem, np.load(filename)])
    
    # Pre-processing
    data_processed = []
    
    for data_i in data:
      data_processed_i, is_nan_flag = process_single_ts(data_i[1], smooth=True, de_trend=True)
      data_processed.append([data_i[0], data_processed_i])
      if is_nan_flag:
        print(f'The signal {data_i[0]} has NaNs and the missing values have been interpolated')
    
    print(f'Number of processed files: {len(filename_list)}')
    print('Pre-processed files: ')
    for index, name in enumerate(filename_list):
        print(index, name)

    update_signals(choose_rows.value, choose_columns.value)
    button_plot.disabled = False
    box_acf.disabled = False
    button_create_df.disabled = False
    button_preprocess_data.disabled = False
    
button_preprocess_data = widgets.Button(
    description='Preprocess data',
    disabled=True,
    button_style='',
    layout=widgets.Layout(width='max-content')
)
button_preprocess_data.on_click(preprocess_data)

In [76]:
display(button_preprocess_data)
display(out_process_data)

Button(description='preprocess data', disabled=True, layout=Layout(width='max-content'), style=ButtonStyle())

Output()

## Step 3. Plot some time series

Enter the number of rows and columns (max 3 x 3) you want to plot in the same window and then select from the dropdown menu filenames of the specific timeseries you want to display. 
Click on the ```Plot now``` to show the figure.
If you want to save the the plot, insert a title (without extension) and click on the ```Save plot``` button. The figure will be automatically saved in the figures folder both in html and png format.

In [77]:
output_signals = widgets.Output()

choose_rows = widgets.BoundedIntText(
    value=1,
    min=1,
    max=4,
    step=1,
    description='Rows:',
    disabled=False
)

choose_columns= widgets.BoundedIntText(
    value=1,
    min=1,
    max=4,
    step=1,
    description='Columns:',
    disabled=False
)

list_of_signal_widget = []
@output_signals.capture(clear_output=True)
def update_signals(rows, cols):
    global list_of_signal_widget
    global filename_list

    list_of_signal_widget = []

    if not filename_list:
        return
    for i in range(rows):
        for j in range(cols):
            list_of_signal_widget.append(
                widgets.Combobox(
                    options=filename_list,
                    placeholder='Select or type a file',
                    description='Signal_(' + str(i) + ',' + str(j) + ')',
                    disabled=False,
                    layout=Layout(width='50%')
                )
            )
        display(widgets.HBox(list_of_signal_widget[i * cols : i * cols + cols]))

interactive_signals = widgets.interactive_output(update_signals, 
                                                 {'rows': choose_rows,
                                                  'cols': choose_columns})

In [78]:
display(widgets.HBox([choose_rows, choose_columns]), interactive_signals)
display(output_signals)

Output()

Output()

In [79]:
output_plot_signals = widgets.Output()
fig_new = None

@output_plot_signals.capture(clear_output=True)
def plot_on_button(b=None):  
    global list_of_signal_widget
    global fig_new

    if all(s.value is None for s in list_of_signal_widget):
        return

    button_plot.disabled = True
    
    rows = choose_rows.value
    columns = choose_columns.value
    t_conversion = 0.097
    
    plot_info = {'y_min': -2.5,
                 'y_max': 2.5,
                }
    colors = ["#43C6DB", "#93FFE8", "dodgerblue", 
              "mediumseagreen", "slateblue", "#737CA1",
              "#29465B", "#368BC1", "#AFDCEC", "#66CDAA"]

    data_processed_plot = [None for _ in range(rows * columns)]
    signal_list_plot = [None for _ in range(rows * columns)]
    for i, s in enumerate(list_of_signal_widget):
        if s.value:
            data_processed_plot[i] = data_processed[filename_list.index(s.value)]
            signal_list_plot[i] = s.value
    
    fig_new = make_subplots(rows=rows, cols=columns, subplot_titles=signal_list_plot)
    r = 1
    c = 1
    for i in range(len(data_processed_plot)): 

        if data_processed_plot[i] is not None:
            fig_ = px.line(x=[t * t_conversion for t in range(len(data_processed_plot[i][1][0]))],
                           y=[data_i for data_i in data_processed_plot[i][1]], 
                           color_discrete_sequence=colors,
                           range_y=[plot_info['y_min'], plot_info['y_max']])
            fig_.update_layout(showlegend=False)
            fig_.update_traces(opacity=0.5)
                
            for trace in fig_["data"]:
              fig_new.add_trace(trace, row=r, col=c)
              fig_new.update_xaxes(title_text='Time [s]', row=r, col=c)
              fig_new.update_yaxes(title_text='y [\u03BCm]', row=r, col=c, range=[plot_info['y_min'], plot_info['y_max']])
            fig_new.update_layout(height=800, width=1000)
            fig_new.update_layout(showlegend=False)
               
        if c < (columns):
            # r = 0
            c += 1
        elif c == (columns):
            r += 1
            c = 1
        
    fig_new.show()
    button_plot.disabled = False

button_plot = widgets.Button(
    description='Plot now',
    disabled=True,
    button_style='', # 'success', 'info', 'warning', 'danger' or ''
    # icon='check'
)
button_plot.on_click(plot_on_button)

In [80]:
display(button_plot)
display(output_plot_signals)

Button(description='plot now', disabled=True, style=ButtonStyle())

Output()

In [81]:
save_plot_title= widgets.Text(
    value='',
    placeholder='Insert plot title',
    layout=Layout(width='400px'),
    description='Plot title:',
    disabled=False   
)

button_save_plot = widgets.Button(
    description='save plot',
    disabled=True,
    button_style='', # 'success', 'info', 'warning', 'danger' or ''
    #icon='check'
)

def change_save_plot_title(b=None):
    if save_plot_title.value == '':
        button_save_plot.disabled = True
    else:
        button_save_plot.disabled = False

save_plot_title.observe(change_save_plot_title, names="value")

def save_plot(b=None):
    if fig_new:
        fig_new.write_html(os.path.join(figure_path.value, f"{save_plot_title.value}.html"))
        fig_new.write_image(os.path.join(figure_path.value, f"{save_plot_title.value}.png"), format='png')
        with output_plot_signals:
            print("Figure saved!")

button_save_plot.on_click(save_plot)

In [82]:
display(widgets.HBox([save_plot_title, button_save_plot]))

## Step 4. Create DataFrame with extracted features and data info

In this section we extract the temporal and create a table with all the data info and the extracted features, and update the working datasheet. The ACF plots are saved in the figure/ACF_plots folder.

- Contraction Power
- Mean
- Percentage of Count Above Threshold
- Percentage of Absolute Sum of Changes
- 75 Quantile
- Standard Deviation
- Fundamental Interval

Here you can set the threshold for the feature *Percentage of Count Above Threshold* and decide if you want to compute the ACF (autocorrelation function) to extract the fundamental interval and create the ACF plots in the figure folder. Then click on the ```Create dataframe```.
If you want to save the table in a csv file in the local directory, choose a name (without extension) and click the ```Save csv``` button.

In [83]:
progress_bar= widgets.FloatProgress(
    value=0,
    min=0,
    max=1,
    description='',
    bar_style=''
)

def update_progress_bar(b=None):
    global computed
    increments = 0.25
    while True:
        if computed:
            progress_bar.value = 1
            progress_bar.description = 'done'
            progress_bar.bar_style = 'success'
            return
        else:
            if progress_bar.value > (1 - increments):
                progress_bar.value = 0
            else:
                progress_bar.value = progress_bar.value + increments
            progress_bar.description = 'computing'
            progress_bar.bar_style = 'warning'
        time.sleep(.5)

In [84]:
widget_threshold = widgets.BoundedFloatText(
    value=0.5,
    min=0.,
    max=1.,
    step=.01,
    description='Threshold',
    disabled=False
)

button_create_df = widgets.Button(
    description='Create dataframe',
    disabled=True,
    button_style='', # 'success', 'info', 'warning', 'danger' or ''
    #icon='check'
)

box_acf = widgets.Checkbox(
    value=True,
    description='Compute ACF',
    disabled=True
)

df_fundamental_interval = None

def extract_acf(b=None):
    global df_fundamental_interval
    global computed
    computed = False
    progress_bar_thread = threading.Thread(target=update_progress_bar)
    progress_bar_thread.start()
    
    df_fundamental_interval = extract_acf_values(data_processed, figure_path.value)
    computed = True
    progress_bar_thread.join()

df_data = None
output_create_df = widgets.Output()

@output_create_df.capture(clear_output=True)
def create_df(b=None):
    global df_data
    
    threshold = widget_threshold.value
    # Feature extraction
    df_data = create_feature_in_dataframe(data_processed, df_fundamental_interval, threshold)
    #Create organoid column
    data_info['Organoid'] = data_info['Video name'].apply(lambda x: os.path.basename(x).split("_")[-3])
    #Create position column
    data_info['Position'] = data_info['Video name'].apply(lambda x: os.path.basename(x).split("_")[-2])
    # Set index to merge the two dataframe, updating the values that are computed from tfresh
    data_info.set_index('Video name', inplace=True)
    df_data.set_index('Video name', inplace=True)
    data_info.update(df_data)
    data_info.reset_index(drop=False, inplace=True)
    # Remove Unnamed 0 colum # Not clear where it is created
    if 'Unnamed: 0' in data_info.columns:
        data_info.drop(columns=['Unnamed: 0'], inplace=True)
    # Update excel file
    data_info.to_excel(excel_file.value)


    update_markers_and_colors()
    update_markers_and_colors_bivariate()
    save_csv_filename.disabled = False
    button_plot_univariate.disabled = False
    button_plot_violin_scatter.disabled = False
    button_plot_bivariate.disabled = False
    
    with output_create_df:
        print('DataFrame created! We show here only the first 5 rows.')
    display(HTML(data_info.head().to_html()))
    

def on_button_clicked(b=None):
    if box_acf.value:
        extract_acf()
    create_df()

button_create_df.on_click(on_button_clicked)

In [85]:
display(widgets.HBox([widget_threshold, box_acf, button_create_df]))
display(progress_bar)
display(output_create_df)

FloatProgress(value=0.0, max=1.0)

Output()

In [86]:
save_csv_filename= widgets.Text(
    value='', #Datasheet_template_example.csv
    placeholder='Insert csv filename',
    layout=Layout(width='400px'),
    description='csv filename: ',
    disabled=True   
)

button_save_csv = widgets.Button(
    description='Save csv',
    disabled=True,
    button_style='', # 'success', 'info', 'warning', 'danger' or ''
    #icon='check'
)

def change_save_csv(b=None):
    if save_csv_filename.value == '':
        button_save_csv.disabled = True
    else:
        button_save_csv.disabled = False

save_csv_filename.observe(change_save_csv, names="value")

def save_csv(b=None):
    data_info.to_csv(save_csv_filename.value + '.csv')

button_save_csv.on_click(save_csv)

In [87]:
display(widgets.HBox([save_csv_filename, button_save_csv]))

## Step 5. Univariate analysis

### Extracted Time Features Boxplots
Focus on one variable: select it from the drop down menu and click on the ```Plot univariate``` button to show the boxplot for the different extracted features.

The figure will be automatically saved in the figures folder.

In [88]:
univariate_x_boxplot = widgets.Dropdown(
    value='Treatment during contraction',
    placeholder='Choose feature',
    options=['Treatment during contraction', 'Phenotype'],
    description='x_boxplot:',
    ensure_option=True,
    disabled=False
)

output_plot_univariate = widgets.Output()

@output_plot_univariate.capture(clear_output=True)
def plot_univariate(b=None):
    button_plot_univariate.disabled = True
    
    fig_uni_variate = make_subplots(rows=2, cols=3)
    ax_list = [[1, 1], [1, 2], [1, 3], [2, 1], [2, 2], [2, 3]]
    ind = 0
    
    extracted_feature_columns = ['Contraction power', 'Mean',
           '%_of_count_above_threshold', '%_of_absolute_sum_of_changes',
           'Quantile_75', 'Standard_deviation', '%_of_count_above_mean']
    for ax, col in zip(ax_list, extracted_feature_columns):
        fig_ = px.box(data_info, x=univariate_x_boxplot.value, y=col, color=univariate_x_boxplot.value)# , points="outliers") # Options: None, all, ouliers, suspectedoutliers")
        for trace in fig_["data"]:
            trace.legendgroup = trace.name
            fig_uni_variate.add_trace(trace, row=ax[0], col=ax[1])
            if ind != 0:
                fig_uni_variate.data[-1].showlegend = False
    
        if col == 'Contraction power':
            units = 'Contraction power [\u03BCm^2 / s]'
        elif col in ['Standard_deviation', 'Mean', 'Quantile_75']:
            units = col + ' [\u03BCm]'
        else:
            units = col
        fig_uni_variate.update_layout(height=800, width=1500)
        fig_uni_variate.update_xaxes(title_text=univariate_x_boxplot.value, row=ax[0], col=ax[1])
        fig_uni_variate.update_yaxes(title_text=units, row=ax[0], col=ax[1])
        ind += 1
    
    fig_uni_variate.show()
    fig_uni_variate.write_html(figure_path.value + "/Univariate - boxplot - " + univariate_x_boxplot.value +".html")
    fig_uni_variate.write_image(figure_path.value + "/Univariate - boxplot - " + univariate_x_boxplot.value + ".png", format='png')

    button_plot_univariate.disabled = False

button_plot_univariate = widgets.Button(
    description='Plot univariate',
    disabled=True,
    button_style='', # 'success', 'info', 'warning', 'danger' or ''
    #icon='check'
)

button_plot_univariate.on_click(plot_univariate)

In [89]:
display(widgets.HBox([univariate_x_boxplot, button_plot_univariate]))
display(output_plot_univariate)

Output()

## Step 6. Violin + Scatter Plot 

### Customize markers and colors 

You can customize your plot by choosing the markers for different video positions and the colors of different organoids.
From the dropdown menu, you can select the extracted feature to display on the y axis, the input variable for the x axis and the grouping colors, choosing between the treatment or the phenotype.

Then click on the ```Plot violin-scatter``` to show the plot.
The figure will be automatically saved in the figures folder.

In [90]:
def update_marker(change):
    category = change.owner.description[9:]
    new_marker = change.new
    dict_markers[category] = new_marker

def update_color(change):
    organoid = change.owner.description.split()[1]
    new_color = change.new
    organoid_color_map[organoid] = new_color

output_markers_and_colors = widgets.Output()

list_of_marker_widget = []
list_of_color_widget = []
list_of_violin_widgets = []
@output_markers_and_colors.capture(clear_output=True)
def update_markers_and_colors(b=None):
    global list_of_marker_widget
    global list_of_color_widget
    global list_of_violin_widgets
    global dict_markers
    global organoid_color_map

    list_of_marker_widget = []
    list_of_color_widget = []
    list_of_violin_widgets = []
    
    # Marker
    display_markdown(''' Select a marker for each **Position**:''', raw=True)
    categories_position = data_info['Position'].unique()
    default_marker = 'circle'
    dict_markers = {}
    
    for c in categories_position:
        w = widgets.Dropdown(
            options=['triangle-up', 'circle', 'diamond', 'cross', 'square', 'circle-open', 'square-open'],
            description='category ' + str(c) ,
            value=default_marker,
            disabled=False,
            style= {'description_width': 'initial'},
            layout=widgets.Layout(width='95%')
        )
        w.observe(update_marker, names='value')
        list_of_marker_widget.append(w)
        display(w)
    
      # Check if category already exists in dictionary
        if c not in dict_markers:
          dict_markers[c] = default_marker
    
    # Color
    display_markdown(''' Select a color for each **Organoid**:''', raw=True)
    organoids = data_info['Organoid'].unique()
    organoid_color_map = {}
    default_color = 'green'
    for organoid in organoids:
        organoid_color_map[organoid] = default_color
    
    color_dropdowns = {}
    
    for organoid in organoids:
        color_picker = widgets.ColorPicker(
            description=f'Organoid {organoid}',
            value=default_color,
            style= {'description_width': 'initial'},
            layout=widgets.Layout(width='95%')
        )
        color_picker.observe(update_color, names='value')
        list_of_color_widget.append(color_picker)
        display(color_picker)

    # x, y and color violin
    display_markdown(''' Choose the feature for the classification:''', raw=True)
    widgets_x_violin_plot = widgets.Dropdown(
        value='Treatment during contraction',
        placeholder='Choose feature',
        options=['Treatment during contraction', 'Phenotype'],
        description='x_violin_plot:',
        ensure_option=True,
        disabled=False,
        style= {'description_width': 'initial'},
        layout=widgets.Layout(width='95%')
    )
    list_of_violin_widgets.append(widgets_x_violin_plot)
    display(widgets_x_violin_plot)
    
    widgets_y_violin_plot = widgets.Dropdown(
        value='Contraction power',
        placeholder='Choose feature',
        options=['Contraction power', '%_of_absolute_sum_of_changes', '%_of_count_above_threshold', 'Quantile_75', 'Standard_deviation', 'Total Area'],
        description='y_violin_plot:',
        ensure_option=True,
        disabled=False,
        style= {'description_width': 'initial'},
        layout=widgets.Layout(width='95%')
    )
    list_of_violin_widgets.append(widgets_y_violin_plot)
    display(widgets_y_violin_plot)
    
    widgets_color_violin = widgets.Dropdown(
        value='Treatment during contraction',
        placeholder='Choose feature',
        options=['Organoid', 'Position', 'Treatment during contraction', 'Phenotype'],
        description='color_violin:',
        ensure_option=True,
        disabled=False,
        style= {'description_width': 'initial'},
        layout=widgets.Layout(width='95%')
    )
    list_of_violin_widgets.append(widgets_color_violin)
    display(widgets_color_violin)

In [91]:
display(output_markers_and_colors)

Output()

In [92]:
output_violin_scatter = widgets.Output()

@output_violin_scatter.capture(clear_output=True)
def plot_violin_scatter(b=None):
    button_plot_violin_scatter.disabled = True
    
    x_violin_plot = list_of_violin_widgets[0].value
    y_violin_plot = list_of_violin_widgets[1].value
    color_violin = list_of_violin_widgets[2].value
    treatment_color_map = {'None': 'lightgrey', 'BRANAPLAM': 'lightgrey', 'RISDIPLAM': 'lightgrey'}
    
    fig = px.violin(data_info, x=x_violin_plot, y=y_violin_plot,
                          color=color_violin,
                          color_discrete_map=treatment_color_map,
                          hover_data=data_info.columns
      )

    # fig.update_layout(height=700, width=1000)
    fig.update_traces(
                showlegend=False,
            )
    
    # Scatter
    fig_scatter = px.scatter(data_info, x=x_violin_plot, y=y_violin_plot,
                     color='Organoid',
                     color_discrete_map=organoid_color_map,
                     symbol='Position',
                     hover_name='Video name',
                     #color_discrete_sequence=color_scheme#px.colors.qualitative.Alphabet
                     )
    
    unique_legends = set()
    
    for trace in fig_scatter.data:
        position = trace.name
        marker_symbol = dict_markers.get(position.split(',')[1][1:], 'circle')
        legend_name = f"{position.split(',')[1][1:]} - {trace.hovertext[0]}"
        treatment = trace.hovertext[0]
        #color = organoid_color_map.get(position.split(',')[0], default_color)
    
        fig_scatter.update_traces(
          marker=dict(symbol=marker_symbol, size=7),
          selector=dict(name=position),
        )
        if legend_name not in unique_legends:
            unique_legends.add(legend_name)
        else:
            fig_scatter.update_traces(
                showlegend=False,
            )
    
    for trace in fig_scatter.data:
      fig.add_trace(trace)
    
    fig.update_layout(scattermode="group", scattergap=0.95) # to be considered? ###Why not working???? It works if I change scatter with bar...double check with whole dataset
    #fig.update_layout(barmode="group", bargroupgap=0.95) # to be considered? ###Why not working???? It works if I change scatter with bar...double check with whole dataset
    
    fig.update_layout(height=800, width=1500)
    fig.update_xaxes(title_text=x_violin_plot)
    fig.update_yaxes(title_text=y_violin_plot)
    
    #@markdown The figure will be automatically saved in the 'figures' folder.
    fig.show()
    fig.write_html(figure_path.value + "/Univariate_violin_scatter_" + x_violin_plot + ".html")
    fig.write_image(figure_path.value + "/Univariate_violin_scatter_ " + x_violin_plot + ".png", format='png')

    button_plot_violin_scatter.disabled = False

button_plot_violin_scatter = widgets.Button(
    description='Plot violin-scatter',
    disabled=True,
    button_style='', # 'success', 'info', 'warning', 'danger' or ''
    #icon='check'
)
button_plot_violin_scatter.on_click(plot_violin_scatter)

In [93]:
display(button_plot_violin_scatter)
display(output_violin_scatter)

Button(description='plot violin-scatter', disabled=True, style=ButtonStyle())

Output()

## Step 7. Bivariate analysis

### Customize markers and colors

You can select markers for different phenotypes and colors for different treatments.

Then click on the ```Plot bivariate``` button to show several combinations of extracted features. The figure will be automatically saved in the figures folder.

In [94]:
def update_marker_bivariate(change):
    category = change.owner.description
    new_marker = change.new
    dict_markers_bivariate[category] = new_marker


def update_color_bivariate(change):
    category = change.owner.description
    new_color = change.new if change.new else default_color
    dict_colors_bivariate[category] = new_color

output_markers_and_colors_bivariate = widgets.Output()

list_of_marker_widget_bivariate = []
list_of_color_widget_bivariate = []
@output_markers_and_colors_bivariate.capture(clear_output=True)
def update_markers_and_colors_bivariate(b=None):
    global list_of_marker_widget_bivariate
    global list_of_color_widget_bivariate
    global dict_markers_bivariate
    global dict_colors_bivariate
    
    #Marker
    display_markdown(''' Select a marker for each **Phenotype**:''', raw=True)
    categories_phenotype = data_info['Phenotype'].unique()
    default_marker = 'circle'
    dict_markers_bivariate = {}
    
    for c in categories_phenotype:
        phenotype_wigdet = widgets.Dropdown(
            options=['circle', 'diamond', 'cross', 'square', 'circle-open', 'square-open'],
            description=str(c),
            value=default_marker,
            disabled=False
        )
        phenotype_wigdet.observe(update_marker_bivariate, names='value')
        display(phenotype_wigdet)
        list_of_marker_widget_bivariate.append(phenotype_wigdet)
        # Check if category already exists in dictionary
        if c not in dict_markers_bivariate:
          dict_markers_bivariate[c] = default_marker

    # Color
    display_markdown(''' Select a color for each **Treatment**:''', raw=True)
    categories_treatment = data_info['Treatment during contraction'].unique()
    default_color = 'blue'
    dict_colors_bivariate = {}
    for c in categories_treatment:
        treatment_widget = widgets.ColorPicker(
            concise=False,
            description=str(c),
            value=default_color,
            disabled=False
        )
        treatment_widget.observe(update_color_bivariate, names='value')
        display(treatment_widget)
        list_of_color_widget_bivariate.append(treatment_widget)
        # Check if category already exists in dictionary
        if c not in dict_colors_bivariate:
            dict_colors_bivariate[c] = default_color
    

In [95]:
display(output_markers_and_colors_bivariate)

Output()

In [96]:
output_bivariate = widgets.Output()

@output_bivariate.capture(clear_output=True)
def plot_bivariate(b=None):
    button_plot_bivariate.disabled = True
    var_plot = [
    ['Contraction power', '%_of_absolute_sum_of_changes', 1, 1],
    ['Contraction power', 'Quantile_75', 1, 2],
    ['%_of_count_above_threshold', '%_of_absolute_sum_of_changes', 1, 3],
    ['%_of_absolute_sum_of_changes', 'Quantile_75', 2, 1],
    ['%_of_absolute_sum_of_changes', 'Standard_deviation', 2, 2],
    ['Quantile_75', 'Standard_deviation', 2, 3]
    ]

    fig_final = make_subplots(rows=2, cols=3)
    
    ind = 0
    unique_legends = set()
    
    for x, y, r, c in var_plot:
        fig_ = px.scatter(data_info, x=x, y=y,
                          color='Treatment during contraction',
                          color_discrete_map=dict_colors_bivariate,
                          symbol='Phenotype',
                          hover_name='Video name'
                          )
    
        for trace in fig_.data:
            phenotype = trace.name
            split_phenotype = phenotype.split(',')
            if len(split_phenotype) > 1:
              marker_symbol = dict_markers_bivariate.get(split_phenotype[1][1:], 'circle')
              legend_name = f"{phenotype.split(',')[1][1:]} - {trace.hovertext[0]}"
            else:
              # Handle the case where there aren't enough elements after splitting
              # For example:
              marker_symbol = 'default_value'
              legend_name = 'deafault_for_this_as_well'
            #marker_symbol = dict_markers.get(phenotype.split(',')[1][1:], 'circle')
            # legend_name = f"{phenotype.split(',')[1][1:]} - {trace.hovertext[0]}"
            treatment = trace.hovertext[0]
            color= dict_colors_bivariate.get(phenotype.split(',')[0])
    
            if legend_name not in unique_legends:
                fig_final.add_trace(
                    go.Scatter(
                        x=trace.x,
                        y=trace.y,
                        mode=trace.mode,
                        name=trace.name,
                        legendgroup=trace.legendgroup,
                        marker=dict(symbol=marker_symbol, color=color),  # Assign color based on treatment
                        hovertext=trace.hovertext
                    ),
                    row=r, col=c
                )
                unique_legends.add(legend_name)
            else:
                fig_final.add_trace(
                    go.Scatter(
                        x=trace.x,
                        y=trace.y,
                        mode=trace.mode,
                        name=trace.name,
                        legendgroup=trace.legendgroup,
                        showlegend=False,
                        marker=dict(symbol=marker_symbol, color=color),  # Assign color based on treatment
                        hovertext=trace.hovertext
                    ),
                    row=r, col=c
                )
    
        fig_final.update_xaxes(title_text=x, row=r, col=c)
        fig_final.update_yaxes(title_text=y, row=r, col=c)
    
    fig_final.update_layout(height=800, width=1500)
    fig_final.write_html(figure_path.value + "/Bivariate.html")
    fig_final.write_image(figure_path.value + "/Bivariate.png")
    fig_final.show()
    button_plot_bivariate.disabled = False

button_plot_bivariate = widgets.Button(
    description='Plot bivariate',
    disabled=True,
    button_style='', # 'success', 'info', 'warning', 'danger' or ''
    #icon='check'
)
button_plot_bivariate.on_click(plot_bivariate)

In [97]:
display(button_plot_bivariate)
display(output_bivariate)

Button(description='plot bivariate', disabled=True, style=ButtonStyle())

Output()